# Лабораторная работа №2. Сравнение групп, корреляция и регрессия

## Контекст задачи

Вы продолжаете работать аналитиком данных в страховой компании **MedGuard** (см. лабораторную №1).

После первого отчёта продакт-менеджер (PM) знает, *кто* наши клиенты. Теперь он хочет понять, **от чего зависят выплаты** и можно ли их **прогнозировать**:

> «Мне не нужны просто средние. Скажи, какие группы клиентов действительно отличаются по выплатам, а какие различия — шум. И дай мне формулу, по которой можно прикинуть ожидаемые выплаты для нового клиента».

**Ваша задача** — сравнить группы клиентов (t-критерий, дисперсионный анализ), найти взаимосвязи (корреляция) и построить модель выплат (линейная регрессия).

В этой версии работы **ничего не нужно считать вручную по формулам**: для каждого метода найдите и примените готовую функцию из библиотек. Главное — правильно выбрать метод, прочитать результат и сделать вывод.

---

## Датасет

Тот же **Medical Cost Personal Dataset**, что и в лабораторной №1. Файл `insurance.csv` должен лежать в одной папке с этим ноутбуком:

- https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv

| Колонка | Описание |
|---|---|
| `age` | возраст застрахованного, лет |
| `sex` | пол: `male` / `female` |
| `bmi` | индекс массы тела (кг/м²) |
| `children` | количество детей, покрытых полисом |
| `smoker` | курит ли клиент: `yes` / `no` |
| `region` | регион: `northeast`, `northwest`, `southeast`, `southwest` |
| `charges` | сумма страховых выплат за год, USD |

---

## Шпаргалка: какие функции пригодятся

| Задача | Функция |
|---|---|
| Доверительный интервал по t- и z-распределению | `stats.t.interval`, `stats.norm.interval`, `stats.sem` |
| Сравнение двух средних | `stats.ttest_ind(a, b, equal_var=False)` |
| Однофакторный дисперсионный анализ | `stats.f_oneway`, `ols(...)` + `sm.stats.anova_lm` |
| Какие именно группы различаются | `pairwise_tukeyhsd` (из `statsmodels.stats.multicomp`) |
| Корреляция | `df.corr()`, `stats.pearsonr` |
| Линейная регрессия | `ols("y ~ x1 + x2", data=df).fit()`, `.summary()`, `.params`, `.pvalues`, `.rsquared_adj`, `.predict()` |
| Графики | `sns.boxplot`, `sns.heatmap`, `sns.scatterplot`, `sns.regplot`, `sm.qqplot` |

## Правила выполнения

1. Уровень значимости: $\alpha = 0.05$.
2. Для каждой проверки гипотезы сформулируйте $H_0$ и $H_1$.
3. После каждого задания — **вывод своими словами**: что получилось и что это значит для бизнеса.
4. Ячейки с `# ваш код` и `**Вывод:**` заполняете вы.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

df = pd.read_csv("insurance.csv")
print(df.shape)
df.head()


## Задание 1. t-распределение: когда данных мало

Отдел продаж принёс выгрузку всего по **15 курильщикам** из нового региона и хочет по ней оценить среднюю выплату. Смоделируем эту ситуацию.

**1.1.** Сделайте выборку `df[df.smoker == "yes"].charges.sample(15, random_state=3)`. Выведите её среднее и стандартную ошибку (`stats.sem`).

**1.2.** Постройте 95% доверительный интервал для среднего двумя функциями: `stats.norm.interval` и `stats.t.interval` (для t укажите `df = n - 1`). Какой интервал шире? Какой из них правильный в этой ситуации и почему?

In [ ]:
# ваш код: выборка из 15 курильщиков, среднее и стандартная ошибка

In [ ]:
# ваш код: 95% ДИ через stats.norm.interval и stats.t.interval

**Вывод:** 

## Задание 2. Сравнение двух средних: t-критерий Стьюдента

Финансовый отдел хочет ввести для курильщиков отдельный тариф. PM просит доказать, что разница в выплатах — не случайность. Заодно маркетинг утверждает, что «мужчины обходятся дороже женщин».

**2.1.** Выведите описательную статистику `charges` по группам `smoker` (`df.groupby("smoker").charges.describe()`) и постройте boxplot.

**2.2.** Сформулируйте $H_0$ и $H_1$ и проверьте их функцией `stats.ttest_ind(..., equal_var=False)`. Что означают полученные `statistic` и `pvalue`?

**2.3.** Проверьте утверждение маркетинга: отличаются ли средние выплаты мужчин и женщин? Затем повторите тот же тест **только среди некурящих**. Посмотрите на долю курильщиков среди мужчин и женщин (`pd.crosstab(df.sex, df.smoker, normalize="index")`) и объясните, что произошло (вспомните из лекции про «третью переменную»).

In [ ]:
# ваш код: описательная статистика по smoker и boxplot

In [ ]:
# ваш код: t-тест курильщики vs некурящие

In [ ]:
# ваш код: мужчины vs женщины (все и только некурящие), доля курильщиков по полу

**Вывод:** 

## Задание 3. Однофакторный дисперсионный анализ: различаются ли регионы?

Руководитель региона `southeast` просит увеличить бюджет: «у нас выплаты выше, чем у всех». Регионов четыре, поэтому нужен **ANOVA**.

**3.1.** Выведите средние `charges` по регионам и постройте boxplot `charges` по `region`.

**3.2.** Сформулируйте $H_0$ и $H_1$. Проведите ANOVA двумя способами: `stats.f_oneway` и таблицей `sm.stats.anova_lm(ols("charges ~ C(region)", data=df).fit())`. Совпадают ли F и p? Какой вывод?

**3.3.** ANOVA не говорит, **какие именно** регионы различаются. Примените критерий Тьюки `pairwise_tukeyhsd(df.charges, df.region)`. Какие пары регионов различаются значимо (`reject = True`)?

**3.4.** Проведите ANOVA для `bmi` по регионам (`stats.f_oneway`) и сравните средние `bmi`. Как это может объяснить различия в выплатах?

In [ ]:
# ваш код: средние по регионам и boxplot

In [ ]:
# ваш код: stats.f_oneway и anova_lm

In [ ]:
# ваш код: критерий Тьюки

In [ ]:
# ваш код: ANOVA для bmi по регионам

**Вывод:** 

## Задание 4. Корреляция: что связано с выплатами?

**4.1.** Постройте матрицу корреляций Пирсона для `age`, `bmi`, `children`, `charges` (`df[[...]].corr()`) и нарисуйте её тепловой картой (`sns.heatmap(..., annot=True)`). Какая переменная сильнее всего связана с `charges`?

**4.2.** Функцией `stats.pearsonr` получите коэффициент корреляции между `age` и `charges` и его p-значение. Посчитайте коэффициент детерминации $R^2 = r^2$ и объясните его смысл.

**4.3.** Постройте диаграмму рассеяния `age` → `charges`, раскрасив точки по `smoker` (`sns.scatterplot(..., hue="smoker")`). Что видно? Посчитайте `stats.pearsonr` между `age` и `charges` **только для некурящих** и сравните с общим. Почему перед расчётом корреляции нужно смотреть на график?

In [ ]:
# ваш код: матрица корреляций и heatmap

In [ ]:
# ваш код: pearsonr для age и charges, R^2

In [ ]:
# ваш код: scatter по smoker; pearsonr только для некурящих

**Вывод:** 

## Задание 5. Регрессия с одной независимой переменной

PM просит «формулу»: сколько в среднем добавляет к выплатам каждый год возраста?

**5.1.** Постройте модель `ols("charges ~ age", data=df).fit()` и выведите `summary()`. Найдите в таблице $b_0$ (Intercept), $b_1$ (age), их p-значения и $R^2$. Запишите уравнение модели и проинтерпретируйте $b_1$.

**5.2.** Нарисуйте данные с линией регрессии (`sns.regplot(x="age", y="charges", data=df)`).

**5.3.** Методом `.predict()` найдите прогноз выплаты для клиента 50 лет.

**5.4.** Проверьте условия применения регрессии: постройте график остатков (`model.resid`) от `age` и QQ-plot остатков (`sm.qqplot(model.resid, line="s")`). Выполняются ли нормальность остатков и гомоскедастичность?

In [ ]:
# ваш код: модель ols и summary

In [ ]:
# ваш код: regplot

In [ ]:
# ваш код: прогноз для 50 лет

In [ ]:
# ваш код: график остатков и QQ-plot

**Вывод:** 

## Задание 6. Множественная регрессия и выбор лучшей модели

**6.1.** Постройте несколько моделей и сравните их по **исправленному** $R^2$ (`.rsquared_adj`):
- `charges ~ age`
- `charges ~ age + bmi`
- `charges ~ age + bmi + smoker`
- `charges ~ age + bmi + smoker + children`
- `charges ~ age + bmi + smoker + children + sex + region`

Соберите результат в одну таблицу. Почему сравнивать нужно именно исправленный $R^2$?

**6.2.** Выберите лучшую модель с точки зрения «качество / простота». Выведите её `summary()` и проинтерпретируйте коэффициенты, особенно при `smoker` (не забудьте: «при прочих равных»).

**6.3.** В полной модели посмотрите на p-значения (`.pvalues`) для `sex` и `region`. Сравните с выводами заданий 2 и 3.

**6.4.** По выбранной модели предскажите выплату для двух клиентов 50 лет с `bmi = 30`: курящего и некурящего.

In [ ]:
# ваш код: сравнение моделей по R^2 и исправленному R^2

In [ ]:
# ваш код: summary выбранной модели

In [ ]:
# ваш код: p-значения в полной модели

In [ ]:
# ваш код: прогноз для курящего и некурящего клиента